# final_version_0718_v3 DataProcess 啟停教學

本文件說明如何在 API 主機上安全地啟動與停止 DataProcess。DataProcess 的作用是持續產生噴塗線模擬感測資料並寫入 PostgreSQL；停止 DataProcess 只會停止新增資料，不會刪除既有資料。


## 1. 使用前提

- 在執行 Docker 與 API 的 Windows 主機上操作。
- 使用 CMD 或 PowerShell。
- V3 專案路徑為 `C:\Users\User\Desktop\final_version_0718_v3`。
- PostgreSQL 使用既有外部 Volume：`final_version_0627_sprayline_pgdata`。
- `START_0718_v3.bat` 預設不會啟動 DataProcess，需使用本文件的指令手動啟動。
- 同一時間只能啟動一個 DataProcess，否則多個產生器會同時寫入資料，使日期與筆數快速增加。


## 2. 進入 V3 專案資料夾

在 CMD 執行：

```cmd
cd /d C:\Users\User\Desktop\final_version_0718_v3
```

確認目前資料夾內有 `docker-compose.yml`：

```cmd
dir
```


## 3. 啟動前檢查

先確認目前是否已有 DataProcess 在執行：

```cmd
docker ps --filter "name=dataprocess" --format "table {{.Names}}\t{{.Status}}"
```

正常的停止狀態只會顯示標題，不應看到任何 `Up` 或 `Restarting` 的 DataProcess。若畫面已有 DataProcess，先不要再啟動新的產生器。

確認既有資料庫 Volume 仍存在：

```cmd
docker volume inspect final_version_0627_sprayline_pgdata
```

輸出應包含：

```text
"Name": "final_version_0627_sprayline_pgdata"
```


## 4. 啟動 V3 DataProcess

執行：

```cmd
docker compose --profile data up -d --build dataprocess
```

此指令會建立或更新 V3 DataProcess 容器，連接現有的 `sprayline_db`，並在背景持續執行。它不會執行 `setup_db.sql`，也不會清除資料庫。


## 5. 確認只有一個 DataProcess

啟動後再次檢查：

```cmd
docker ps --filter "name=dataprocess" --format "table {{.Names}}\t{{.Status}}"
```

預期只看到一個容器：

```text
final_version_0718_v3-dataprocess-1    Up ...
```

若同時看到其他 0620、0623、0627、0716、0718_v2 等舊版 DataProcess，應停止舊容器，避免重複產生資料：

```cmd
docker update --restart=no <舊容器名稱>
docker stop <舊容器名稱>
```

`<舊容器名稱>` 必須替換成畫面中實際顯示的完整名稱。


## 6. 查看 DataProcess 即時紀錄

執行：

```cmd
docker compose --profile data logs -f --tail 50 dataprocess
```

若持續出現資料生成或寫入訊息，代表 DataProcess 正在運作。按 `Ctrl+C` 只會離開紀錄畫面，不會停止容器。


## 7. 確認資料正在增加

查詢每個站別的最新時間與資料筆數：

```cmd
docker exec sprayline_db psql -U postgres -d sprayline -c "SELECT station_id, COUNT(*) AS row_count, MAX(ts) AS latest_time FROM sensor_1min GROUP BY station_id ORDER BY station_id;"
```

等待一段時間後再次執行。若 `row_count` 與 `latest_time` 增加，表示 DataProcess 已成功寫入 DB。由於這是模擬資料產生器，資料時間可能比實際日期更快向後推進。


## 8. 停止 DataProcess

在 V3 資料夾執行：

```cmd
docker compose --profile data stop dataprocess
```

此操作只停止資料生成，不會停止 API、Manager UI、Engineer UI 或 PostgreSQL，也不會刪除已經寫入的資料。

停止後確認：

```cmd
docker ps --filter "name=dataprocess" --format "table {{.Names}}\t{{.Status}}"
```

畫面不應再出現執行中的 DataProcess。


## 9. 確認停止後資料不再增加

停止後執行兩次以下查詢，兩次之間等待一段時間：

```cmd
docker exec sprayline_db psql -U postgres -d sprayline -c "SELECT station_id, COUNT(*) AS row_count, MAX(ts) AS latest_time FROM sensor_1min GROUP BY station_id ORDER BY station_id;"
```

兩次結果相同，代表 DataProcess 已完全停止。既有資料仍會保留在 PostgreSQL Volume 中。


## 10. 常用狀態檢查

查看主要服務：

```cmd
docker compose ps
```

查看所有 DataProcess，包括已停止的舊容器：

```cmd
docker ps -a --filter "name=dataprocess" --format "table {{.Names}}\t{{.Status}}"
```

查看 V3 DataProcess 最近 100 行紀錄：

```cmd
docker compose --profile data logs --tail 100 dataprocess
```


## 11. 資料保護注意事項

以下指令可能刪除資料或建立全新資料庫，不可在保留既有資料的情況下執行：

```text
docker compose down -v
docker volume rm final_version_0627_sprayline_pgdata
database/setup_db.sql
任何 reset、DROP TABLE、TRUNCATE 或重新初始化資料庫的指令
```

安全停止 DataProcess 應使用：

```cmd
docker compose --profile data stop dataprocess
```


## 12. 完整操作流程

```text
進入 final_version_0718_v3
        ↓
確認沒有舊 DataProcess 運行
        ↓
確認 final_version_0627_sprayline_pgdata 存在
        ↓
使用 --profile data 啟動 V3 DataProcess
        ↓
確認只有一個 V3 DataProcess
        ↓
查看 logs 與資料庫最新時間
        ↓
需要固定測試資料時停止 DataProcess
        ↓
確認資料不再增加且既有資料仍保留
```
